# 多模态模型

此笔记本来自 [AI for Beginners Curriculum](http://aka.ms/ai-beginners)。

## 试验 CLIP

[CLIP](https://openai.com/blog/clip/) 是 OpenAI 发布的公开模型，您可以尝试将其用于不同任务，包括零样本图像分类。请注意，它对资源的需求非常高！


In [2]:
import sys
!{sys.executable} -m pip install git+https://github.com/openai/CLIP.git

我们将首先确保可以使用 GPU（如果可用），然后加载 CLIP 模型。


In [8]:
import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
np.set_printoptions(precision=2,suppress=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

让我们从[Oxford-IIIT 数据集](https://www.robots.ox.ac.uk/~vgg/data/pets/)中获取一部分猫的图像：


In [4]:
!wget https://mslearntensorflowlp.blob.core.windows.net/data/oxcats.tar.gz
!tar xfz oxcats.tar.gz
!rm oxcats.tar.gz

### 零样本图像分类

CLIP 的主要功能是将图像与文本提示进行匹配。例如，如果我们有一张猫的图片，然后尝试将其与文本提示“猫”、“企鹅”、“熊”进行匹配——第一个选项的概率通常会更高。因此，我们可以得出结论，这是一只猫。我们不需要训练模型，因为它已经在一个庞大的数据集上进行了预训练——因此被称为**零样本**。


In [7]:
image = preprocess(Image.open("oxcats/Maine_Coon_1.jpg")).unsqueeze(0).to(device)
text = clip.tokenize(["a penguin", "a bear", "a cat"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)

Label probs: [[0. 0. 1.]]


### 智能图像搜索

在前面的例子中，我们有一张图片和三个文本提示。我们可以在不同的场景中使用CLIP，例如，我们可以拍摄多张猫的照片，然后选择最符合文本描述的那张图片：


In [16]:
cats_img = [ Image.open(os.path.join("oxcats",x)) for x in os.listdir("oxcats") ] 
cats = torch.cat([ preprocess(i).unsqueeze(0) for i in cats_img ]).to(device)
text = clip.tokenize(["a very fat gray cat"]).to(device)
with torch.no_grad():
    logits_per_image, logits_per_text = model(cats, text)
    res = logits_per_text.softmax(dim=-1).argmax().cpu().numpy()

print("Img Index:", res)

plt.imshow(cats_img[res])

Img Index: 97


## 要点

预训练的 CLIP 模型可以用于执行诸如常见物体的图像分类等任务，而无需进行特定领域的训练。此外，它还允许更灵活的分类/图像搜索，能够考虑图片中物体的空间布局。

想了解 CLIP 的另一个令人兴奋的用法，可以查看 **VQGAN+CLIP**。



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。应以原始语言的文档作为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
